# Group 5: feature processing notebook

Этот ноутбук предназначен для исследовательской обработки признаков группы `5`.

Логика ячеек специально повторяет структуру `src/features/group_5/feature_processor.py`,
чтобы позже перенос был почти механическим.

In [21]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

In [22]:
# Пути относительно папки notebooks/group_5/
DATA_PATH = Path("../../data/raw/MIPT_hackathon_dataset.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_5.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_5")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [23]:
df = pd.read_csv(DATA_PATH)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5383 entries, 0 to 5382
Data columns (total 96 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   lead_id                                5383 non-null   str    
 1   sale_ts                                5383 non-null   int64  
 2   sale_date                              5383 non-null   str    
 3   buyout_flag                            5294 non-null   object 
 4   outcome_unknown                        5383 non-null   bool   
 5   handed_to_delivery_ts                  5260 non-null   float64
 6   issued_or_pvz_ts                       5208 non-null   float64
 7   received_ts                            4278 non-null   float64
 8   rejected_ts                            950 non-null    float64
 9   returned_ts                            691 non-null    float64
 10  days_sale_to_handed                    5260 non-null   float64
 11  days_handed_to_

In [24]:

feature_names = load_feature_names_from_txt(FEATURES_PATH)
block_df = df[feature_names].copy()

print("Block shape:", block_df.shape)
display(block_df.head())

Block shape: (5383, 19)


,lead_Ширина,lead_Линейная высота (см),lead_Вид оплаты,returned_ts,lead_Служба доставки,lead_Компания Отправитель,lead_Ответственный за доставку,lead_is_deleted,lead_WIDTH,lead_Почтовый индекс,lead_group_id,lead_Масса (гр),lead_closed_at,contact_created_at,lead_utm_medium,lead_Категория и варианты выбора,received_ts,lead_Модель телефона,lead_id
0,21.0,NaN,Наложенный платеж,NaN,СДЭК до ПВЗ,ООО ТехПродЗдрав,NaN,False,NaN,NaN,546538,NaN,1.763537e+09,1761943124,NaN,S,1.762602e+09,Не удалось узнать,LEAD_0217
1,30.0,NaN,Наложенный платеж,NaN,СДЭК до ПВЗ,ООО ТехПродЗдрав,NaN,False,NaN,NaN,700242,NaN,1.763540e+09,1761906675,cpc,S,1.762789e+09,Смартфон,LEAD_0220
2,30.0,NaN,Наложенный платеж,NaN,СДЭК до ПВЗ,ООО ТехПродЗдрав,NaN,False,NaN,NaN,700242,NaN,1.763537e+09,1739328654,NaN,S,1.762749e+09,Смартфон,LEAD_0058
3,30.0,NaN,Наложенный платеж,1.764676e+09,СДЭК до ПВЗ,ООО ТехПродЗдрав,NaN,False,NaN,NaN,700242,NaN,1.770729e+09,1761975991,NaN,S,NaN,Смартфон,LEAD_0221
4,30.0,NaN,Наложенный платеж,NaN,СДЭК до Двери,ООО ТехПродЗдрав,NaN,False,NaN,NaN,0,NaN,1.763361e+09,1760961350,cpc,D,1.762583e+09,Не удалось узнать,LEAD_0218


## Функции обработки признаков

Названия функций совпадают с private-методами из `feature_processor.py`.

In [12]:
def _add_width_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Ширина" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Ширина"], errors="coerce")
    result["lead_Ширина"] = series.fillna(series.median())

In [13]:
def _add_linear_height_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Линейная высота (см)" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Линейная высота (см)"], errors="coerce")
    result["lead_Линейная высота (см)"] = series.fillna(series.median())
    result["lead_Линейная высота (см)__was_missing"] = series.isna().astype(int)

In [15]:
def _add_payment_type_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Вид оплаты" not in block_df.columns:
        return
    series = (
        block_df["lead_Вид оплаты"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
        .str.lower()
    )
    result["lead_Вид оплаты"] = series.replace({"": "unknown"}).astype(str)

In [16]:
def _add_returned_ts_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "returned_ts" not in block_df.columns:
        return
    ts = pd.to_datetime(block_df["returned_ts"], errors="coerce")
    result["returned_ts"] = ts.astype("string")
    result["returned_ts__is_present"] = ts.notna().astype(int)

In [17]:
def _add_delivery_service_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Служба доставки" not in block_df.columns:
        return
    series = (
        block_df["lead_Служба доставки"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )
    value_counts = series.value_counts(dropna=False)
    rare_values = value_counts[value_counts < 10].index
    result["lead_Служба доставки"] = series.replace(list(rare_values), "OTHER").astype(str)

In [18]:
def _add_default_feature(block_df: pd.DataFrame, result: pd.DataFrame, column: str) -> None:
    series = block_df[column]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        result[column] = series.fillna("").astype(str)
    else:
        result[column] = series

In [19]:
X_block = pd.DataFrame(index=block_df.index)

for column in block_df.columns:
    if column == "lead_Ширина":
        _add_width_feature(block_df, X_block)
    elif column == "lead_Линейная высота (см)":
        _add_linear_height_feature(block_df, X_block)
    elif column == "lead_Вид оплаты":
        _add_payment_type_feature(block_df, X_block)
    elif column == "returned_ts":
        _add_returned_ts_feature(block_df, X_block)
    elif column == "lead_Служба доставки":
        _add_delivery_service_feature(block_df, X_block)
    else:
        _add_default_feature(block_df, X_block, column)

print("Processed shape:", X_block.shape)
display(X_block.head())

Processed shape: (5383, 21)


,lead_Ширина,lead_Линейная высота (см),lead_Линейная высота (см)__was_missing,lead_Вид оплаты,returned_ts,returned_ts__is_present,lead_Служба доставки,lead_Компания Отправитель,lead_Ответственный за доставку,lead_is_deleted,...,lead_Почтовый индекс,lead_group_id,lead_Масса (гр),lead_closed_at,contact_created_at,lead_utm_medium,lead_Категория и варианты выбора,received_ts,lead_Модель телефона,lead_id
0,21.0,9.0,1,наложенный платеж,<NA>,0,СДЭК до ПВЗ,ООО ТехПродЗдрав,,False,...,NaN,546538,NaN,1.763537e+09,1761943124,,S,1.762602e+09,Не удалось узнать,LEAD_0217
1,30.0,9.0,1,наложенный платеж,<NA>,0,СДЭК до ПВЗ,ООО ТехПродЗдрав,,False,...,NaN,700242,NaN,1.763540e+09,1761906675,cpc,S,1.762789e+09,Смартфон,LEAD_0220
2,30.0,9.0,1,наложенный платеж,<NA>,0,СДЭК до ПВЗ,ООО ТехПродЗдрав,,False,...,NaN,700242,NaN,1.763537e+09,1739328654,,S,1.762749e+09,Смартфон,LEAD_0058
3,30.0,9.0,1,наложенный платеж,1970-01-01 00:00:01.764676322,1,СДЭК до ПВЗ,ООО ТехПродЗдрав,,False,...,NaN,700242,NaN,1.770729e+09,1761975991,,S,NaN,Смартфон,LEAD_0221
4,30.0,9.0,1,наложенный платеж,<NA>,0,СДЭК до Двери,ООО ТехПродЗдрав,,False,...,NaN,0,NaN,1.763361e+09,1760961350,cpc,D,1.762583e+09,Не удалось узнать,LEAD_0218


In [20]:
X_block.to_csv(OUTPUT_DIR / "X_block.csv", index=False)
feature_spec = create_feature_spec_template(X_block)
feature_spec.to_csv(OUTPUT_DIR / "feature_spec.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "X_block.csv")
print(OUTPUT_DIR / "feature_spec.csv")

Saved:
..\..\notebook_outputs\group_5\X_block.csv
..\..\notebook_outputs\group_5\feature_spec.csv
